# COP509 Notebook 2: Extraction, Alignment And Classification

This notebook is the assessed Task 2 notebook. It demonstrates recommendation extraction, response-unit extraction, alignment, classification, confidence metadata, validation evidence and an optional interactive inspection demo.

The notebook reads the final validated outputs from project-relative paths and does not overwrite `final_recommendations_246.json`, `evaluation_predictions.csv`, or any source data.

## Google Colab setup

If this notebook is opened through the GitHub Colab link, run the setup cells below first. They clone the full repository so the notebook can access `src/`, `data/`, `outputs/` and the PDF files. Local Jupyter/VS Code users can run the notebook normally.


In [ ]:
# Colab setup: clone the full repository so src/, data/ and outputs/ are available.
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_dir = Path("/content/cop509-public-inquiry-nlp")
    if not repo_dir.exists():
        !git clone https://github.com/DanielLawrence04/cop509-public-inquiry-nlp.git {repo_dir}
    os.chdir(repo_dir)
    print(f"Running from: {Path.cwd()}")
else:
    print("Not running in Colab; using the local repository path.")


In [ ]:
# Install required packages when running in Colab.
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !{sys.executable} -m pip install -r requirements.txt
else:
    print("Local environment detected; skipping Colab pip install.")


In [1]:
from __future__ import annotations

import json
import logging
import sys
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 140)

# Suppress harmless optional model-library noise while preserving real exceptions.
warnings.filterwarnings("ignore", category=FutureWarning, module=r"transformers(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"huggingface_hub(\.|$)")
warnings.filterwarnings("ignore", message=r".*resume_download.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=r".*Torch was not compiled with flash attention.*")
for logger_name in ("transformers", "sentence_transformers", "huggingface_hub"):
    logging.getLogger(logger_name).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging

    transformers_logging.set_verbosity_error()
except Exception:
    pass


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root from the repository layout, not a machine-specific path."""
    start = (start or Path.cwd()).resolve()
    required_dirs = ("src", "data", "outputs")
    coursework_names = ("COP509_Coursework.pdf", "COP509_coursework.pdf")
    for candidate in [start, *start.parents]:
        has_required_dirs = all((candidate / name).is_dir() for name in required_dirs)
        has_coursework_file = any((candidate / name).exists() for name in coursework_names)
        if has_required_dirs and (has_coursework_file or (candidate / "outputs" / "final_recommendations_246.json").exists()):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Expected directories: src/, data/, outputs/ "
        "and either the final export or the COP509 coursework PDF."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATA_RAW_DIR = DATA_DIR / "raw"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FINAL_EXPORT_PATH = OUTPUTS_DIR / "final_recommendations_246.json"
EVALUATION_PREDICTIONS_PATH = OUTPUTS_DIR / "evaluation_predictions.csv"
QA_MATRIX_PATH = DATA_DIR / "ground_truth" / "qa_matrix_queries.json"

# Backward-compatible alias used by older display cells.
RAW_DIR = DATA_RAW_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.getLogger("src.preprocessing").setLevel(logging.ERROR)
logging.getLogger("src.pdf_loader").setLevel(logging.ERROR)
print(f"Project root detected: {PROJECT_ROOT.name}")
print(f"Raw PDF directory: {DATA_RAW_DIR.relative_to(PROJECT_ROOT)}")
print(f"Final export: {FINAL_EXPORT_PATH.relative_to(PROJECT_ROOT)}")


Project root detected: cop509-coursework
Raw PDF directory: data\raw
Final export: outputs\final_recommendations_246.json


## 1. Coursework Task Overview

Task 2 converts paired report and government-response PDFs into recommendation-level rows. In the current final output, this means **246 recommendation rows across the full 8-pair / 16-PDF corpus**. For each row the project records the extracted recommendation, matched response text, classification label, confidence values and validation metadata.

The implementation is more than a minimal rules demo: it combines preset-aware recommendation extraction, response-unit extraction, structured response matching, classification, confidence scoring, final export validation and automated QA/regression tests used during development.

Full manual labels are not available for all 246 rows. This notebook therefore reports validation checks, row counts, label distributions, confidence summaries, inspected examples and a representative Task 2c expected-vs-actual sample. Accuracy and F1 should only be calculated if a full 246-row manual benchmark is added later.

## 2. Dataset And Pair Setup

The shared preset map is the source of truth for the full available corpus. It identifies **8 document pairs / 16 PDFs**: coursework-given pairs plus extension pairs used to test the extraction, alignment and classification workflow.

In [2]:
from backend.core.presets import PRESETS, validate_preset_files

preset_rows = []
for preset in PRESETS.values():
    try:
        validate_preset_files(preset)
        files_ok, notes = True, ""
    except FileNotFoundError as exc:
        files_ok, notes = False, str(exc)
    special = []
    if preset.allow_response_heading_recommendation_fallback:
        special.append("response-heading fallback")
    if preset.inline_recommendation_chapter_prefix:
        special.append(f"inline prefix {preset.inline_recommendation_chapter_prefix}")
    if preset.select_committee_conclusions_section:
        special.append("select-committee section")
    preset_rows.append({
        "preset_id": preset.id,
        "label": preset.label,
        "dataset_group": preset.dataset_group,
        "policy_pdf": preset.policy_pdf.name,
        "response_pdf": preset.response_pdf.name,
        "files_ok": files_ok,
        "special_extraction": ", ".join(special),
        "notes": notes,
    })

preset_df = pd.DataFrame(preset_rows)
if not preset_df["files_ok"].all():
    display(preset_df)
    raise FileNotFoundError("One or more preset PDFs are missing.")
all_preset_pdf_names = sorted({path.name for preset in PRESETS.values() for path in (preset.policy_pdf, preset.response_pdf)})
print(f"Full available preset corpus: {len(PRESETS)} document pairs / {len(all_preset_pdf_names)} PDFs")
print("Task 2 final output is reported across the full 8-pair corpus below.")
display(preset_df)

Full available preset corpus: 8 document pairs / 16 PDFs
Task 2 final output is reported across the full 8-pair corpus below.


,preset_id,label,dataset_group,policy_pdf,response_pdf,files_ok,special_extraction,notes
0,behaviour_change,Behaviour Change,coursework_given,Behaviour-Change-Report-Recomm.pdf,Behaviour-Change-Response.pdf,True,,
1,post_office,Post Office Horizon Inquiry,coursework_given,PostOfficeHorizon-I- Inquiry-Recomm.pdf,PostOfficeHorizon-IT-Inquiry-Response.pdf,True,,
2,space_economy,The Space Economy,coursework_given,TheSpaceEconomyReport.pdf,TheSpaceEconomyResponse.pdf,True,,
3,covid_inquiry,UK Covid-19 Inquiry Module 1,coursework_given,UK-Covid-19-Inquiry-Module-1-Recomm.pdf,UK-Covid-19_Inquiry_Module_1_Response.pdf,True,,
4,blood_inquiry,Infected Blood Inquiry,coursework_given,Volume_1-Blood-Inquiry-Recomm.pdf,Volume_1-Blood-Inquiry-Response.pdf,True,,
5,grenfell_phase2,Grenfell Tower Inquiry — Phase 2,extra_found,Grenfell-Phase2-Volume7-Recomm.pdf,Grenfell-Phase2-Response.pdf,True,"response-heading fallback, inline prefix 113",
6,covid_inquiry_module2,UK Covid-19 Inquiry Module 2,extra_found,UK-Covid-19-Inquiry-Module-2-Recomm.pdf,UK-Covid-19_Inquiry_Module_2_Response.pdf,True,,
7,summer_2024_disorder,Police response to the 2024 summer disorder,extra_found,Summer2024-Disorder-Recomm.pdf,Summer2024-Disorder-Response.pdf,True,select-committee section,


## 3. Source-Code Mirroring

The notebook imports the same shared project modules used by the final validator. Notebook-local code is limited to display helpers and summary tables.

In [3]:
from src.pdf_loader import extract_pages
from src.extraction import extract_recommendations
from src.response_units import extract_response_units
from src.alignment import match_recommendations_to_response_units
from src.classification import classify_response, classify_with_confidence, normalize_label
from scripts.validate_recommendation_export import (
    classification_failures,
    metadata_failures,
    recommendation_cleanup_failures,
    response_leakage_failures,
)

TASK2_LABELS = ["accepted", "partial", "rejected", "not_addressed"]
LABEL_DISPLAY = {
    "accepted": "accepted",
    "partial": "partially accepted",
    "partially_accepted": "partially accepted",
    "rejected": "rejected",
    "not_addressed": "not addressed",
}
print("Loaded source modules for loading, extraction, response units, alignment and classification.")

Loaded source modules for loading, extraction, response units, alignment and classification.


## 4. Final Validated Export Baseline

The selected evidence baseline is the canonical project-relative file `outputs/final_recommendations_246.json`. The supporting classified export is `outputs/evaluation_predictions.csv`. The cell below reads the JSON baseline, checks the expected row counts and leaves the source outputs unchanged.

In [4]:
EXPECTED_PAIR_COUNTS = {
    "behaviour_change": 33,
    "post_office": 19,
    "space_economy": 40,
    "covid_inquiry": 10,
    "blood_inquiry": 58,
    "grenfell_phase2": 58,
    "covid_inquiry_module2": 19,
    "summer_2024_disorder": 9,
}
EXPECTED_CLASS_COUNTS = {"accepted": 104, "partial": 139, "rejected": 1, "not_addressed": 2}
FINAL_EXPORT_CANDIDATES = [FINAL_EXPORT_PATH]


def load_export(path: Path) -> list[dict]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = payload.get("recommendations", payload) if isinstance(payload, dict) else payload
    if not isinstance(rows, list):
        raise ValueError(f"{path} does not contain a recommendation list")
    return rows

selected_export_path, final_rows = None, None
for candidate in FINAL_EXPORT_CANDIDATES:
    if not candidate.exists():
        continue
    rows = load_export(candidate)
    if (
        len(rows) == 246
        and dict(Counter(str(r.get("document_pair")) for r in rows)) == EXPECTED_PAIR_COUNTS
        and dict(Counter(str(r.get("classification")) for r in rows)) == EXPECTED_CLASS_COUNTS
    ):
        selected_export_path, final_rows = candidate, rows
        break

if final_rows is None:
    found = [str(path) for path in FINAL_EXPORT_CANDIDATES if path.exists()]
    raise FileNotFoundError("No validated 246-row final export found. Expected: " + str(FINAL_EXPORT_PATH.relative_to(PROJECT_ROOT)))

final_df = pd.json_normalize(final_rows, sep=".")
print("Selected final export evidence baseline:")
print(selected_export_path.relative_to(PROJECT_ROOT))
print("Canonical evidence name used in this notebook: final_recommendations_246.json")
print(f"Rows loaded: {len(final_df)}")

Selected final export evidence baseline:
outputs\final_recommendations_246.json
Canonical evidence name used in this notebook: final_recommendations_246.json
Rows loaded: 246


## 5. Final Full Export Summary

The compact export label `partial` means partially accepted.

In [5]:
pair_summary = pd.DataFrame({
    "document_pair": list(EXPECTED_PAIR_COUNTS),
    "expected_rows": list(EXPECTED_PAIR_COUNTS.values()),
    "actual_rows": [int((final_df["document_pair"] == pair).sum()) for pair in EXPECTED_PAIR_COUNTS],
})
pair_summary["matches_expected"] = pair_summary["expected_rows"] == pair_summary["actual_rows"]

class_counts = final_df["classification"].value_counts().to_dict()
class_summary = pd.DataFrame({
    "export_label": TASK2_LABELS,
    "display_label": [LABEL_DISPLAY[label] for label in TASK2_LABELS],
    "expected_rows": [EXPECTED_CLASS_COUNTS[label] for label in TASK2_LABELS],
    "actual_rows": [int(class_counts.get(label, 0)) for label in TASK2_LABELS],
})
class_summary["percentage"] = (class_summary["actual_rows"] / len(final_df) * 100).round(1)
class_summary["matches_expected"] = class_summary["expected_rows"] == class_summary["actual_rows"]

print(f"Final row count: {len(final_df)}")
display(pair_summary)
display(class_summary)

Final row count: 246


,document_pair,expected_rows,actual_rows,matches_expected
0,behaviour_change,33,33,True
1,post_office,19,19,True
2,space_economy,40,40,True
3,covid_inquiry,10,10,True
4,blood_inquiry,58,58,True
5,grenfell_phase2,58,58,True
6,covid_inquiry_module2,19,19,True
7,summer_2024_disorder,9,9,True


,export_label,display_label,expected_rows,actual_rows,percentage,matches_expected
0,accepted,accepted,104,104,42.3,True
1,partial,partially accepted,139,139,56.5,True
2,rejected,rejected,1,1,0.4,True
3,not_addressed,not addressed,2,2,0.8,True


## 6. Confidence, Metadata And Validation Checks

The final export includes overall confidence plus debug fields for alignment confidence, lexical similarity, classification confidence, classifier method, alignment method and confidence factors.

In [6]:
confidence_summary = pd.DataFrame([
    {"metric": "overall confidence", "mean": final_df["confidence"].mean(), "min": final_df["confidence"].min(), "max": final_df["confidence"].max()},
    {"metric": "alignment confidence", "mean": final_df["debug.alignment_confidence"].mean(), "min": final_df["debug.alignment_confidence"].min(), "max": final_df["debug.alignment_confidence"].max()},
    {"metric": "classification confidence", "mean": final_df["debug.classification_confidence"].mean(), "min": final_df["debug.classification_confidence"].min(), "max": final_df["debug.classification_confidence"].max()},
]).round(3)
method_summary = final_df["debug.alignment_method"].value_counts(dropna=False).rename_axis("alignment_method").reset_index(name="rows")
classifier_summary = final_df["debug.classifier_method"].value_counts(dropna=False).rename_axis("classifier_method").reset_index(name="rows")
display(confidence_summary)
display(method_summary)
display(classifier_summary)


def duplicate_id_failures(rows: list[dict]) -> dict[str, list[str]]:
    by_pair = defaultdict(list)
    for row in rows:
        by_pair[str(row.get("document_pair"))].append(str(row.get("id")))
    return {pair: sorted([item for item, count in Counter(ids).items() if count > 1]) for pair, ids in by_pair.items() if any(count > 1 for count in Counter(ids).values())}


def blank_id_failures(rows: list[dict]) -> list[tuple[str, str]]:
    return [(str(r.get("document_pair")), str(r.get("id"))) for r in rows if r.get("id") is None or str(r.get("id")).strip() == ""]

validation_df = pd.DataFrame([
    {"check": "row count", "status": "ok" if len(final_rows) == 246 else "failed", "details": len(final_rows)},
    {"check": "document-pair counts", "status": "ok" if pair_summary["matches_expected"].all() else "failed", "details": "see pair summary"},
    {"check": "classification distribution", "status": "ok" if class_summary["matches_expected"].all() else "failed", "details": "see class summary"},
    {"check": "duplicate IDs", "status": "ok" if not duplicate_id_failures(final_rows) else "failed", "details": duplicate_id_failures(final_rows) or "none"},
    {"check": "blank IDs", "status": "ok" if not blank_id_failures(final_rows) else "failed", "details": blank_id_failures(final_rows) or "none"},
    {"check": "response leakage", "status": "ok" if not response_leakage_failures(final_rows) else "failed", "details": response_leakage_failures(final_rows) or "none"},
    {"check": "recommendation cleanup", "status": "ok" if not recommendation_cleanup_failures(final_rows) else "failed", "details": recommendation_cleanup_failures(final_rows) or "none"},
    {"check": "metadata completeness", "status": "ok" if not metadata_failures(final_rows) else "failed", "details": metadata_failures(final_rows) or "none"},
    {"check": "classification corrections", "status": "ok" if not classification_failures(final_rows) else "failed", "details": classification_failures(final_rows) or "none"},
])
display(validation_df)
if not (validation_df["status"] == "ok").all():
    raise AssertionError("One or more final export validation checks failed.")

,metric,mean,min,max
0,overall confidence,0.935,0.8,0.95
1,alignment confidence,0.938,0.9,0.95
2,classification confidence,0.787,0.6,0.92


,alignment_method,rows
0,exact_label,148
1,structured_grouped,52
2,structure,30
3,structured_ordinal,10
4,structured_section,6


,classifier_method,rows
0,rule_based,246


,check,status,details
0,row count,ok,246
1,document-pair counts,ok,see pair summary
2,classification distribution,ok,see class summary
3,duplicate IDs,ok,none
4,blank IDs,ok,none
5,response leakage,ok,none
6,recommendation cleanup,ok,none
7,metadata completeness,ok,none
8,classification corrections,ok,none


## 7. Live Pipeline Demonstration On One Pair

The final export above is loaded read-only. This section runs the shared project modules on one representative pair so the extraction, response-unit and alignment stages are visible without changing final outputs.

In [7]:
DEMO_PRESET_ID = "behaviour_change"
demo_preset = PRESETS[DEMO_PRESET_ID]
policy_pages = extract_pages(demo_preset.policy_pdf)
response_pages = extract_pages(demo_preset.response_pdf)
demo_recommendations = extract_recommendations(
    policy_pages,
    response_pages_fallback=(response_pages if demo_preset.allow_response_heading_recommendation_fallback else None),
    inline_chapter_prefix=demo_preset.inline_recommendation_chapter_prefix,
    select_committee_section=demo_preset.select_committee_conclusions_section,
)
demo_response_units = extract_response_units(response_pages)
demo_alignments = match_recommendations_to_response_units(demo_recommendations, demo_response_units, top_k=3, similarity_threshold=0.05)
print(f"Demo preset: {DEMO_PRESET_ID} - {demo_preset.label}")
print(f"Policy pages: {len(policy_pages)}")
print(f"Response pages: {len(response_pages)}")
print(f"Recommendations extracted: {len(demo_recommendations)}")
print(f"Response units extracted: {len(demo_response_units)}")
print(f"Alignment candidates returned: {len(demo_alignments)}")

Demo preset: behaviour_change - Behaviour Change
Policy pages: 109
Response pages: 18
Recommendations extracted: 33
Response units extracted: 22
Alignment candidates returned: 99


## 8. Recommendation Extraction And Response Matching

Recommendation extraction records method and confidence. Response-unit matching returns alignment method, similarity and alignment confidence.

In [8]:
demo_rec_df = pd.DataFrame(demo_recommendations)
display(demo_rec_df["extraction_method"].value_counts().rename_axis("extraction_method").reset_index(name="rows"))
display(demo_rec_df[["rec_id", "item_label", "page_number", "extraction_method", "confidence", "text"]].assign(text=lambda df: df["text"].str.slice(0, 180) + "...").head(8))

unit_df = pd.DataFrame(demo_response_units)
if not unit_df.empty:
    unit_df = unit_df.assign(response_preview=unit_df["response_text"].fillna("").str.slice(0, 160) + "...")
    display(unit_df[["recommendation_labels", "page_start", "page_end", "extraction_confidence", "boundary_reason", "response_preview"]].head(8))

demo_align_df = pd.DataFrame(demo_alignments)
if not demo_align_df.empty:
    display(demo_align_df.groupby("match_method", as_index=False).agg(rows=("rec_id", "count"), mean_similarity=("similarity", "mean"), mean_alignment_confidence=("alignment_confidence", "mean")).round(3))
    display(demo_align_df[["rec_id", "match_method", "similarity", "alignment_confidence", "page_number", "matched_text"]].assign(matched_text=lambda df: df["matched_text"].str.slice(0, 180) + "...").head(8))

,extraction_method,rows
0,structured section,33


,rec_id,item_label,page_number,extraction_method,confidence,text
0,0,8.1,67,structured section,0.86,"The idea of the Government intervening to change people’s behaviour will often be controversial, and so it is important that ministers a..."
1,1,8.2,67,structured section,0.86,There is a lack of applied research at a population level to support specific interventions to change the behaviour of large groups of p...
2,2,8.3,67,structured section,0.86,We acknowledge that there will be occasions when it is legitimate for a government not to implement behaviour change interventions for w...
3,3,8.4,67,structured section,0.92,"We agree with the principle, stated in the Government’s Principles of Scientific Advice , that ministers should explain publicly their r..."
4,4,8.5,67,structured section,0.92,We urge ministers to consult their departmental Chief Scientific Advisers about whether the amount of money spent on applied behaviour c...
5,5,8.6,68,structured section,0.92,"We recommend that, at the earliest opportunity, the Government appoint a Chief Social Scientist who reports to the Government Chief Scie..."
6,6,8.7,68,structured section,0.92,"We further recommend that the Government consider whether existing mechanisms for the provision of social scientific advice, in particul..."
7,7,8.8,68,structured section,0.92,"Departmental Chief Scientific Advisers, whether or not they have experience of the sciences of human behaviour, should be responsible fo..."


,recommendation_labels,page_start,page_end,extraction_confidence,boundary_reason,response_preview
0,[8.1],3,3,0.95,next_heading,The Government agrees with the Committee‘s conclusion. It is important that behavioural interventions draw from the available evidence b...
1,[8.2],3,3,0.95,next_heading,"The Government agrees with the Committee‘s conclusion. As with all policy decisions, we believe that it is important that the Government..."
2,"[8.3, 8.4]",4,4,0.75,multi_label_block,The Government agrees with the Committee‘s conclusion. As with all policy decisions and in line with the GCSA Guidelines on the use of s...
3,[8.5],4,4,0.95,next_heading,The Government agrees with the need to ensure that Ministers are able to consult departmental experts when considering the amount of spe...
4,"[8.6, 8.7]",4,5,0.75,multi_label_block,It is essential that policy making is informed by high quality evidence drawing upon the full range of analytical disciplines. The Gover...
5,"[8.8, 8.9]",5,6,0.75,multi_label_block,The Government agrees that the best expertise and advice from across all the relevant disciplines are brought to bear on decisions in th...
6,"[8.10, 8.11]",6,6,0.75,multi_label_block,"The Cabinet Office has already published a paper, jointly with the Institute for Government, which sets out an evidence-based framework ..."
7,"[8.12, 8.13]",6,7,0.75,multi_label_block,We agree that it is important to ensure that effective mechanisms exist for sharing and disseminating knowledge about behavioural interv...


,match_method,rows,mean_similarity,mean_alignment_confidence
0,exact_label,33,0.950,0.950
1,semantic,66,0.127,0.127


,rec_id,match_method,similarity,alignment_confidence,page_number,matched_text
0,0,exact_label,0.950000,0.950000,3,The Government agrees with the Committee‘s conclusion. It is important that behavioural interventions draw from the available evidence b...
1,0,semantic,0.128381,0.128381,11,"The Government agrees with the Committee‘s recommendation. However, we would emphasise that Change4Life presents a number of challenges ..."
2,0,semantic,0.126074,0.126074,10,The Government agrees that it is important for local authorities to have the ability to tailor their interventions to local populations....
3,1,exact_label,0.950000,0.950000,3,"The Government agrees with the Committee‘s conclusion. As with all policy decisions, we believe that it is important that the Government..."
4,1,semantic,0.130277,0.130277,8,"As the Committee has acknowledged, collaboration between Government, public\nhealth, commercial and voluntary organisations presents a u..."
5,1,semantic,0.123090,0.123090,18,The Department for Transport drew local authorities‘ attention to a wide range of research and evaluation evidence when the Local Sustai...
6,2,exact_label,0.950000,0.950000,4,The Government agrees with the Committee‘s conclusion. As with all policy decisions and in line with the GCSA Guidelines on the use of s...
7,2,semantic,0.131071,0.131071,11,"The Government agrees with the Committee‘s recommendation. However, we would emphasise that Change4Life presents a number of challenges ..."


## 9. Classification Labels

The classifier returns `accepted`, `partially_accepted`, `rejected` or `not_addressed`. The final export normalises `partially_accepted` to the compact `partial` value.

In [9]:
alignments_by_rec = defaultdict(list)
for alignment in demo_alignments:
    alignments_by_rec[int(alignment["rec_id"])].append(dict(alignment))

classified_demo_rows = []
for rec in demo_recommendations:
    rec_matches = sorted(alignments_by_rec.get(int(rec["rec_id"]), []), key=lambda item: float(item.get("similarity", 0.0)), reverse=True)
    best = rec_matches[0] if rec_matches else None
    if best:
        raw_label, class_conf = classify_with_confidence(str(best.get("matched_text") or ""))
        label = normalize_label(raw_label)
    else:
        label, class_conf = "not_addressed", 0.0
    export_label = "partial" if label == "partially_accepted" else label
    classified_demo_rows.append({
        "item_label": rec.get("item_label"),
        "classification": LABEL_DISPLAY.get(export_label, export_label),
        "classification_confidence": round(float(class_conf), 3),
        "alignment_confidence": round(float((best or {}).get("alignment_confidence", 0.0) or 0.0), 3),
        "match_method": (best or {}).get("match_method"),
        "recommendation_preview": str(rec.get("text", ""))[:120] + "...",
        "response_preview": str((best or {}).get("matched_text", ""))[:120] + "...",
    })
classified_demo_df = pd.DataFrame(classified_demo_rows)
display(classified_demo_df.head(10))
display(classified_demo_df["classification"].value_counts().rename_axis("classification").reset_index(name="rows"))

,item_label,classification,classification_confidence,alignment_confidence,match_method,recommendation_preview,response_preview
0,8.1,accepted,0.78,0.95,exact_label,"The idea of the Government intervening to change people’s behaviour will often be controversial, and so it is important ...",The Government agrees with the Committee‘s conclusion. It is important that behavioural interventions draw from the avai...
1,8.2,accepted,0.85,0.95,exact_label,There is a lack of applied research at a population level to support specific interventions to change the behaviour of l...,"The Government agrees with the Committee‘s conclusion. As with all policy decisions, we believe that it is important tha..."
2,8.3,accepted,0.75,0.95,exact_label,We acknowledge that there will be occasions when it is legitimate for a government not to implement behaviour change int...,The Government agrees with the Committee‘s conclusion. As with all policy decisions and in line with the GCSA Guidelines...
3,8.4,accepted,0.75,0.95,exact_label,"We agree with the principle, stated in the Government’s Principles of Scientific Advice , that ministers should explain ...",The Government agrees with the Committee‘s conclusion. As with all policy decisions and in line with the GCSA Guidelines...
4,8.5,accepted,0.85,0.95,exact_label,We urge ministers to consult their departmental Chief Scientific Advisers about whether the amount of money spent on app...,The Government agrees with the need to ensure that Ministers are able to consult departmental experts when considering t...
5,8.6,partially accepted,0.70,0.95,exact_label,"We recommend that, at the earliest opportunity, the Government appoint a Chief Social Scientist who reports to the Gover...",It is essential that policy making is informed by high quality evidence drawing upon the full range of analytical discip...
6,8.7,partially accepted,0.70,0.95,exact_label,We further recommend that the Government consider whether existing mechanisms for the provision of social scientific adv...,It is essential that policy making is informed by high quality evidence drawing upon the full range of analytical discip...
7,8.8,accepted,0.92,0.95,exact_label,"Departmental Chief Scientific Advisers, whether or not they have experience of the sciences of human behaviour, should b...",The Government agrees that the best expertise and advice from across all the relevant disciplines are brought to bear on...
8,8.9,accepted,0.92,0.95,exact_label,"We recommend that the Cabinet Secretary, in consultation with the Government Chief Scientific Adviser and Chief Social S...",The Government agrees that the best expertise and advice from across all the relevant disciplines are brought to bear on...
9,8.10,accepted,0.85,0.95,exact_label,"We recommend that the Cabinet Office, in consultation with the Chief Social Scientist, once appointed, consider how to c...","The Cabinet Office has already published a paper, jointly with the Institute for Government, which sets out an evidence-..."


,classification,rows
0,partially accepted,19
1,accepted,14


## 10. Task 2c Evaluation Note

A full manual ground-truth label set is not available for all 246 final rows. For that reason, this notebook does not report overall accuracy, precision, recall or F1 for Task 2. Instead, it reports final row counts, label distributions, validation checks, confidence summaries, inspected examples and the representative expected-vs-actual Task 2c sample below.

If a complete 246-row manual benchmark is added later, accuracy and F1 can be calculated against that benchmark. The existing `labels.json` file is not a full-row benchmark and is not used for that purpose.

## Task 2c Evaluation: Expected vs Actual Sample

A full 246-row manual benchmark is not available, so this section uses a representative manually reviewed sample across document pairs, alignment patterns and classification labels. It is intended to address the Task 2c expected-vs-actual requirement without claiming full-corpus accuracy, precision, recall or F1.

The sample covers accepted, partial, rejected and not addressed outcomes, including easy exact matches, grouped responses, numbering/hard cases and known limitations that should remain in future regression checks.

In [10]:
task2c_sample_spec = [
    {
        "document_pair": "behaviour_change",
        "recommendation_id": "8.1",
        "expected_response_summary": "Government agrees that behaviour-change interventions need an explainable evidence base.",
        "system_response_summary": "Matched response says the Government agrees and explains use of evidence for behaviour-change policy.",
        "expected_classification": "accepted",
        "alignment_match": "yes",
        "comment": "Easy exact-numbered match with clear agreement language.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "behaviour_change",
        "recommendation_id": "8.25",
        "expected_response_summary": "Government disagrees with the committee's view but gives an alternative position on responsibility pledges.",
        "system_response_summary": "Matched response says the Government does not agree with the committee's view and explains its alternative position.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Mixed disagreement plus explanation is treated as partial rather than simple rejection.",
        "suggested_fix_if_error": "Keep as a regression case for disagreement-with-alternative wording.",
    },
    {
        "document_pair": "behaviour_change",
        "recommendation_id": "8.33",
        "expected_response_summary": "Government does not set car-use carbon targets but describes wider carbon-reduction policy.",
        "system_response_summary": "Matched response rejects specific targets while referring to broader economy-wide carbon-reduction targets.",
        "expected_classification": "partial",
        "alignment_match": "partial",
        "comment": "Known limitation: nearby repeated response headings make exact numbering alone fragile, although the final row is usable.",
        "suggested_fix_if_error": "Add stronger heading disambiguation and keep this as a hard-case regression test.",
    },
    {
        "document_pair": "post_office",
        "recommendation_id": "4",
        "expected_response_summary": "DBT broadly accepts legal-support funding but limits scope and capacity.",
        "system_response_summary": "Matched response says DBT broadly accepts and explains limits on funded legal advice.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Clear partial acceptance language with qualifications.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "post_office",
        "recommendation_id": "13",
        "expected_response_summary": "DBT rejects closing the Dispute Resolution Procedure for current claimants.",
        "system_response_summary": "Matched response explicitly says DBT rejects the recommendation.",
        "expected_classification": "rejected",
        "alignment_match": "yes",
        "comment": "Explicit rejection example.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "space_economy",
        "recommendation_id": "23",
        "expected_response_summary": "Government explores support for Earth observation collaboration rather than committing to the full recommendation.",
        "system_response_summary": "Matched response discusses exploring opportunities to support the UK EO sector and collaboration.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Structured response match with exploratory commitment language.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "space_economy",
        "recommendation_id": "30",
        "expected_response_summary": "Government notes space is already critical infrastructure and describes access-to-space policy.",
        "system_response_summary": "Matched response states space is already designated as Critical National Infrastructure and explains the policy context.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Hard case where section-heading cleanup avoids recommendation-text leakage.",
        "suggested_fix_if_error": "Keep cleanup checks for trailing headings in future exports.",
    },
    {
        "document_pair": "space_economy",
        "recommendation_id": "69",
        "expected_response_summary": "Government describes international space-agency agreements but does not provide the requested complete annual publication.",
        "system_response_summary": "Matched response lists international partnership agreements and explains current practice.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Structured ordinal response match.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "covid_inquiry",
        "recommendation_id": "2",
        "expected_response_summary": "Government agrees to a greater Cabinet Office role but retains the lead department model.",
        "system_response_summary": "Matched response agrees with more Cabinet Office leadership while keeping the Lead Government Department model.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Good example of partial implementation against a multi-part recommendation.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "covid_inquiry",
        "recommendation_id": "8",
        "expected_response_summary": "Government agrees with publishing reports on preparedness and resilience.",
        "system_response_summary": "Matched response agrees with transparency and public understanding of risks and mitigation.",
        "expected_classification": "accepted",
        "alignment_match": "yes",
        "comment": "Exact label match with clear agreement language.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "blood_inquiry",
        "recommendation_id": "4a i",
        "expected_response_summary": "UK nations accept introducing or supporting statutory candour duties.",
        "system_response_summary": "Matched grouped response says the recommendation is accepted in full by relevant administrations.",
        "expected_classification": "accepted",
        "alignment_match": "yes",
        "comment": "Grouped/nested response case; label preservation is important.",
        "suggested_fix_if_error": "Keep grouped-response unit tests for nested labels.",
    },
    {
        "document_pair": "blood_inquiry",
        "recommendation_id": "4c ii",
        "expected_response_summary": "Governments accept aspects in principle/full but leave implementation detail conditional.",
        "system_response_summary": "Matched grouped response combines accepted-in-principle and accepted-in-full wording across administrations.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Grouped response with mixed commitments across jurisdictions.",
        "suggested_fix_if_error": "Review jurisdiction-specific sublabels if a finer benchmark is added.",
    },
    {
        "document_pair": "grenfell_phase2",
        "recommendation_id": "31",
        "expected_response_summary": "Inspectorate accepts inspection/reporting recommendation for London Fire Brigade.",
        "system_response_summary": "Matched response says HMICFRS accepts the recommendation and describes inspection work.",
        "expected_classification": "accepted",
        "alignment_match": "yes",
        "comment": "Clear accepted response in a long inquiry document.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "grenfell_phase2",
        "recommendation_id": "50",
        "expected_response_summary": "Government supports the local-authority recommendation but frames delivery through guidance and local constraints.",
        "system_response_summary": "Matched response supports the recommendation and notes local-authority implementation constraints.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Qualified support is correctly not treated as full acceptance.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "grenfell_phase2",
        "recommendation_id": "58",
        "expected_response_summary": "Government accepts reconsidering the LGA Guide advice and says new guidance will be published.",
        "system_response_summary": "Matched response accepts the recommendation and notes the old advice was redacted and new guidance is planned.",
        "expected_classification": "accepted",
        "alignment_match": "yes",
        "comment": "Hard case from embedded inquiry recommendations; final row is correctly aligned.",
        "suggested_fix_if_error": "Keep inline-recommendation extraction checks for Grenfell-style prose.",
    },
    {
        "document_pair": "covid_inquiry_module2",
        "recommendation_id": "1",
        "expected_response_summary": "UK government states the recommendation is for Northern Ireland rather than for UK government response.",
        "system_response_summary": "Matched response says this recommendation is not for the UK government to respond to.",
        "expected_classification": "not_addressed",
        "alignment_match": "yes",
        "comment": "Not-addressed example caused by responsibility outside UK government response scope.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "covid_inquiry_module2",
        "recommendation_id": "13",
        "expected_response_summary": "UK government again states the Northern Ireland recommendation is not for it to answer.",
        "system_response_summary": "Matched response says this recommendation is not for the UK government to respond to.",
        "expected_classification": "not_addressed",
        "alignment_match": "yes",
        "comment": "Second not-addressed example with exact numbering.",
        "suggested_fix_if_error": "None - correct in sample.",
    },
    {
        "document_pair": "summer_2024_disorder",
        "recommendation_id": "1",
        "expected_response_summary": "CPS/Government response describes media protocol work but does not fully satisfy the update cadence requested.",
        "system_response_summary": "Matched response discusses the media protocol and information-sharing arrangements after Southport.",
        "expected_classification": "partial",
        "alignment_match": "yes",
        "comment": "Select-committee extraction case; conclusion-only items are excluded upstream.",
        "suggested_fix_if_error": "Keep select-committee section-boundary checks for future documents.",
    },
]

def _safe_text_column(df: pd.DataFrame, column: str) -> pd.Series:
    if column in df.columns:
        return df[column].astype(str)
    return pd.Series([""] * len(df), index=df.index, dtype="string")

def _lookup_final_row(document_pair: str, recommendation_id: str) -> dict:
    pair_col = _safe_text_column(final_df, "document_pair")
    id_col = _safe_text_column(final_df, "id")
    matches = final_df[(pair_col == str(document_pair)) & (id_col == str(recommendation_id))]
    if matches.empty:
        return {}
    return matches.iloc[0].to_dict()

task2c_rows = []
for item in task2c_sample_spec:
    actual = _lookup_final_row(item["document_pair"], item["recommendation_id"])
    actual_classification = str(actual.get("classification", "missing"))
    expected_classification = item["expected_classification"]
    task2c_rows.append({
        "document_pair": item["document_pair"],
        "recommendation_id": item["recommendation_id"],
        "expected_response_summary": item["expected_response_summary"],
        "system_response_summary": item["system_response_summary"],
        "expected_classification": expected_classification,
        "actual_classification": actual_classification,
        "alignment_match": item["alignment_match"],
        "classification_match": actual_classification == expected_classification,
        "comment": item["comment"],
        "suggested_fix_if_error": item["suggested_fix_if_error"],
    })

task2c_eval_df = pd.DataFrame(task2c_rows)
task2c_columns = [
    "document_pair",
    "recommendation_id",
    "expected_response_summary",
    "system_response_summary",
    "expected_classification",
    "actual_classification",
    "alignment_match",
    "classification_match",
    "comment",
    "suggested_fix_if_error",
]
display(task2c_eval_df[task2c_columns])

label_coverage = task2c_eval_df["expected_classification"].value_counts().rename_axis("label").reset_index(name="sample_rows")
display(label_coverage)
print(f"Task 2c representative sample rows: {len(task2c_eval_df)}")
print("Full-corpus accuracy/F1 is not reported because a complete manual benchmark is unavailable.")

,document_pair,recommendation_id,expected_response_summary,system_response_summary,expected_classification,actual_classification,alignment_match,classification_match,comment,suggested_fix_if_error
0,behaviour_change,8.1,Government agrees that behaviour-change interventions need an explainable evidence base.,Matched response says the Government agrees and explains use of evidence for behaviour-change policy.,accepted,accepted,yes,True,Easy exact-numbered match with clear agreement language.,None - correct in sample.
1,behaviour_change,8.25,Government disagrees with the committee's view but gives an alternative position on responsibility pledges.,Matched response says the Government does not agree with the committee's view and explains its alternative position.,partial,partial,yes,True,Mixed disagreement plus explanation is treated as partial rather than simple rejection.,Keep as a regression case for disagreement-with-alternative wording.
2,behaviour_change,8.33,Government does not set car-use carbon targets but describes wider carbon-reduction policy.,Matched response rejects specific targets while referring to broader economy-wide carbon-reduction targets.,partial,partial,partial,True,"Known limitation: nearby repeated response headings make exact numbering alone fragile, although the final row is usable.",Add stronger heading disambiguation and keep this as a hard-case regression test.
3,post_office,4,DBT broadly accepts legal-support funding but limits scope and capacity.,Matched response says DBT broadly accepts and explains limits on funded legal advice.,partial,partial,yes,True,Clear partial acceptance language with qualifications.,None - correct in sample.
4,post_office,13,DBT rejects closing the Dispute Resolution Procedure for current claimants.,Matched response explicitly says DBT rejects the recommendation.,rejected,rejected,yes,True,Explicit rejection example.,None - correct in sample.
5,space_economy,23,Government explores support for Earth observation collaboration rather than committing to the full recommendation.,Matched response discusses exploring opportunities to support the UK EO sector and collaboration.,partial,partial,yes,True,Structured response match with exploratory commitment language.,None - correct in sample.
6,space_economy,30,Government notes space is already critical infrastructure and describes access-to-space policy.,Matched response states space is already designated as Critical National Infrastructure and explains the policy context.,partial,partial,yes,True,Hard case where section-heading cleanup avoids recommendation-text leakage.,Keep cleanup checks for trailing headings in future exports.
7,space_economy,69,Government describes international space-agency agreements but does not provide the requested complete annual publication.,Matched response lists international partnership agreements and explains current practice.,partial,partial,yes,True,Structured ordinal response match.,None - correct in sample.
8,covid_inquiry,2,Government agrees to a greater Cabinet Office role but retains the lead department model.,Matched response agrees with more Cabinet Office leadership while keeping the Lead Government Department model.,partial,partial,yes,True,Good example of partial implementation against a multi-part recommendation.,None - correct in sample.
9,covid_inquiry,8,Government agrees with publishing reports on preparedness and resilience.,Matched response agrees with transparency and public understanding of risks and mitigation.,accepted,accepted,yes,True,Exact label match with clear agreement language.,None - correct in sample.


,label,sample_rows
0,partial,10
1,accepted,5
2,not_addressed,2
3,rejected,1


Task 2c representative sample rows: 18
Full-corpus accuracy/F1 is not reported because a complete manual benchmark is unavailable.


## 11. Examples And Hard Cases

These static tables keep representative evidence visible in PDF export.

In [11]:
example_keys = {
    ("behaviour_change", "8.1"): "accepted example",
    ("post_office", "4"): "partially accepted example",
    ("post_office", "13"): "rejected example",
    ("covid_inquiry_module2", "1"): "not addressed example",
    ("covid_inquiry_module2", "13"): "not addressed example",
}
example_df = final_df[final_df.apply(lambda row: (str(row["document_pair"]), str(row["id"])) in example_keys, axis=1)].copy()
example_df["evidence_note"] = example_df.apply(lambda row: example_keys[(str(row["document_pair"]), str(row["id"]))], axis=1)
example_df["recommendation_preview"] = example_df["recommendation_text"].fillna("").str.slice(0, 150) + "..."
example_df["response_preview"] = example_df["matched_response_text"].fillna("").str.slice(0, 150) + "..."
display(example_df[["evidence_note", "document_pair", "id", "classification", "confidence", "debug.alignment_method", "debug.alignment_confidence", "debug.classification_confidence", "recommendation_preview", "response_preview"]].sort_values(["document_pair", "id"]))

hard_notes = {
    ("behaviour_change", "8.33"): "response heading repeats 8.32, so exact numbering is insufficient",
    ("space_economy", "30"): "trailing section-heading cleanup prevents recommendation leakage",
    ("blood_inquiry", "4a i"): "nested labels are preserved for grouped response structure",
    ("grenfell_phase2", "58"): "inline/fallback extraction handles embedded inquiry recommendations",
    ("summer_2024_disorder", "1"): "select-committee extraction excludes conclusion-only items",
}
hard_df = final_df[final_df.apply(lambda row: (str(row["document_pair"]), str(row["id"])) in hard_notes, axis=1)].copy()
hard_df["case_note"] = hard_df.apply(lambda row: hard_notes[(str(row["document_pair"]), str(row["id"]))], axis=1)
hard_df["recommendation_preview"] = hard_df["recommendation_text"].fillna("").str.slice(0, 150) + "..."
hard_df["response_preview"] = hard_df["matched_response_text"].fillna("").str.slice(0, 150) + "..."
display(hard_df[["case_note", "document_pair", "id", "classification", "confidence", "debug.alignment_method", "recommendation_page", "matched_response_page", "recommendation_preview", "response_preview"]].sort_values(["document_pair", "id"]))

,evidence_note,document_pair,id,classification,confidence,debug.alignment_method,debug.alignment_confidence,debug.classification_confidence,recommendation_preview,response_preview
0,accepted example,behaviour_change,8.1,accepted,0.95,exact_label,0.95,0.78,"The idea of the Government intervening to change people’s behaviour will often be controversial, and so it is important that ministers a...",The Government agrees with the Committee‘s conclusion. It is important that behavioural interventions draw from the available evidence b...
218,not addressed example,covid_inquiry_module2,1,not_addressed,0.85,exact_label,0.95,0.90,Recommendation 1: Chief Medical Officer for Northern Ireland The Department of Health (Northern Ireland) should reconstitute the role of...,This recommendation is not for the UK government to respond to....
230,not addressed example,covid_inquiry_module2,13,not_addressed,0.85,exact_label,0.95,0.90,Recommendation 13: Amendment of the Ministerial Code in Northern Ireland The Executive Office should amend the Ministerial Code to impos...,This recommendation is not for the UK government to respond to....
45,rejected example,post_office,13,rejected,0.95,exact_label,0.95,0.85,The current Dispute Resolution Procedure in HSS should be closed once all claimants currently within the Procedure have either (a) settl...,DBT rejects this recommendation as we believe it comes into conflict with the principle of “full and fair” redress set out as we accepte...
36,partially accepted example,post_office,4,partial,0.95,exact_label,0.95,0.80,All claimants in HSS shall be entitled to obtain legal advice funded by the Department prior to choosing between accepting the Fixed Sum...,DBT broadly accepts this recommendation. DBT accepts that some HSS claimants require access to legal support on their decision to take t...


,case_note,document_pair,id,classification,confidence,debug.alignment_method,recommendation_page,matched_response_page,recommendation_preview,response_preview
32,"response heading repeats 8.32, so exact numbering is insufficient",behaviour_change,8.33,partial,0.95,exact_label,72,18,We recommend that the Government (a) establish and publish targets for a reduction in carbon emissions as a result of a reduction in car...,The Government does not believe that it is appropriate to set specific targets for reducing carbon emissions by reducing car use. The Go...
109,nested labels are preserved for grouped response structure,blood_inquiry,4a i,accepted,0.90,structured_grouped,288,30,A statutory duty of candour in healthcare should be introduced in Northern Ireland....,This recommendation is accepted in full by the Northern Ireland Executive This recommendation is accepted in full by the Scottish Govern...
217,inline/fallback extraction handles embedded inquiry recommendations,grenfell_phase2,58,accepted,0.95,exact_label,248,30,that the advice contained in paragraph 79.11 of the LGA Guide be reconsidered....,The government accepts this recommendation. The advice contained in paragraph 79.11 of the LGA Guide was redacted in 2021. The Home Offi...
66,trailing section-heading cleanup prevents recommendation leakage,space_economy,30,partial,0.95,structure,107/108,16,"If the Government wishes to progress with the pursuit of sovereign launch capability, it should consider designating UK spaceports as Cr...","The Government is committed to securing assured access to space for the UK, which requires access to both a spaceport and launch vehicle..."
237,select-committee extraction excludes conclusion-only items,summer_2024_disorder,1,partial,0.95,exact_label,38,4,"Notwithstanding potential changes to contempt of court laws, we recommend that the CPS publish its new media protocol as soon as possibl...",The CPS supported the committee to scrutinise the process by which information was shared with the public following the appalling attack...


## 12. Optional Interactive Extraction And Alignment Demo

Interactive widgets are optional. The static evidence above is sufficient for PDF marking: it includes the final 246-row summary, pair counts, classification distribution, validation checks, the Task 2c expected-vs-actual sample, representative examples and hard cases. When `ipywidgets` is unavailable or a notebook viewer cannot render widget state, the cells below also provide static fallback tables.


In [ ]:
LABEL_COLOURS = {
    "accepted": "#198754",
    "partial": "#fd7e14",
    "partially_accepted": "#fd7e14",
    "rejected": "#dc3545",
    "not_addressed": "#6c757d",
}

try:
    from src.widgets.extraction_explorer import show as show_extraction_explorer

    display(Markdown("Interactive extraction explorer loaded. If your notebook viewer cannot render widgets, use the static fallback table below."))
    show_extraction_explorer(demo_recommendations, demo_alignments, classify_response, LABEL_COLOURS)
except Exception as exc:
    display(Markdown(
        f"Interactive extraction explorer unavailable in this environment (`{type(exc).__name__}: {exc}`). "
        "Static demo alignment rows are shown below for marking."
    ))

fallback_alignment_df = pd.DataFrame(demo_alignments)
if fallback_alignment_df.empty:
    fallback_alignment_df = pd.DataFrame(demo_recommendations)
preview_cols = [
    col for col in [
        "recommendation_id",
        "id",
        "response_id",
        "classification",
        "confidence",
        "alignment_confidence",
        "method",
        "recommendation_text",
        "matched_response_text",
    ]
    if col in fallback_alignment_df.columns
]
display(fallback_alignment_df[preview_cols].head(10) if preview_cols else fallback_alignment_df.head(10))


### Optional Final-Row Inspector

This small inspector is separate from the widget above. It lets a live notebook user choose any final export row and view the recommendation, matched response, classification, confidence and provenance. If widgets are unavailable, the cell displays a static sample of final export rows instead.


In [ ]:
import html
from IPython.display import HTML, display

final_export_df = final_df.reset_index(drop=True).copy()


def _row_label(row: pd.Series) -> str:
    return f"{row['document_pair']} {row['id']} - {LABEL_DISPLAY.get(row['classification'], row['classification'])}"


def _render_final_row(row_index: int) -> HTML:
    row = final_export_df.iloc[int(row_index)]
    recommendation = html.escape(str(row.get('recommendation_text', '') or ''))
    response = html.escape(str(row.get('matched_response_text', '') or ''))
    classification = html.escape(str(LABEL_DISPLAY.get(row.get('classification'), row.get('classification'))))
    confidence = html.escape(str(row.get('confidence', '')))
    align_method = html.escape(str(row.get('debug.alignment_method', '')))
    align_conf = html.escape(str(row.get('debug.alignment_confidence', '')))
    response_page = html.escape(str(row.get('matched_response_page', '')))
    row_id = html.escape(f"{row.get('document_pair')} {row.get('id')}")
    return HTML(f"""
    <div style="border:1px solid #ddd;padding:10px;border-radius:6px;font-family:sans-serif;">
      <div><strong>Recommendation ID:</strong> {row_id}</div>
      <div><strong>Classification:</strong> {classification}</div>
      <div><strong>Confidence:</strong> {confidence}</div>
      <div><strong>Alignment method:</strong> {align_method} (confidence {align_conf})</div>
      <div><strong>Matched response page:</strong> {response_page}</div>
      <hr style="border:none;border-top:1px solid #eee;margin:8px 0;"/>
      <div><strong>Recommendation text</strong></div>
      <div style="max-height:180px;overflow:auto;white-space:pre-wrap;">{recommendation}</div>
      <div style="margin-top:8px;"><strong>Matched response text</strong></div>
      <div style="max-height:220px;overflow:auto;white-space:pre-wrap;">{response}</div>
    </div>
    """)

try:
    import ipywidgets as widgets

    row_dropdown = widgets.Dropdown(
        options=[(_row_label(row), idx) for idx, row in final_export_df.iterrows()],
        value=0,
        description="Row",
        layout=widgets.Layout(width="100%"),
    )
    row_output = widgets.Output()

    def _update_row(change=None) -> None:
        with row_output:
            row_output.clear_output(wait=True)
            display(_render_final_row(row_dropdown.value))

    row_dropdown.observe(_update_row, names="value")
    _update_row()
    display(widgets.VBox([row_dropdown, row_output]))
except Exception as exc:
    display(Markdown(
        f"Final-row widget unavailable in this environment (`{type(exc).__name__}: {exc}`). "
        "A static sample of final export rows is shown below."
    ))
    static_cols = [
        "document_pair",
        "id",
        "classification",
        "confidence",
        "debug.alignment_method",
        "debug.alignment_confidence",
        "recommendation_text",
        "matched_response_text",
    ]
    static_cols = [col for col in static_cols if col in final_export_df.columns]
    display(final_export_df[static_cols].head(10))


## 13. Challenges, Optional Web Interface And Limitations

**Challenges and resolutions.** The workflow handles unreliable numbering, varied recommendation layouts, quoted recommendation text inside responses, nested labels and incomplete manual ground truth. It uses documented preset-specific settings where needed and preserves method/confidence metadata so hard cases can be inspected.

**Automated QA evidence.** The `tests/` folder contains regression checks for response-unit extraction, structured matching, classification corrections, final export validation and the frontend evaluation-summary helpers. These tests were used to repeatedly check known hard cases while refining the extraction/alignment pipeline.

**Optional web interface.** The FastAPI/Vite interface exposes the same project workflow interactively, with screenshots in `screenshots/` as supporting evidence. The final submission is local-first because the full corpus, optional model setup and pipeline outputs are better suited to local marking than limited free hosting.

**Limitations and future work.** Add full 246-row manual labels for accuracy/F1, keep expanding hard-case tests for new document layouts, improve OCR quality auditing and consider a second-stage semantic reranker for weak matches after submission.

## 14. Final Conclusion

The final Task 2 evidence shows 246 recommendation-level rows across the full corpus of eight document pairs / 16 PDFs, with the expected pair counts and classification distribution. It covers Task 2a recommendation extraction, Task 2b response matching/classification and Task 2c evaluation through validation checks plus a representative expected-vs-actual sample, while avoiding any claim of full-corpus manual accuracy. The notebook uses the shared project modules, preserves confidence metadata, references the final JSON/CSV exports, provides static and optional interactive evidence, and avoids overwriting final outputs.